# Two species of invariant harmonic residue

### the omitted-cycle mechanism, and the traversal class identified

The $\S19$ counterexample's surviving $\mathcal K_4$-harmonic class was reported earlier as a
candidate **odd-cycle blind spot**. That reading is refuted here and replaced with a measured
dichotomy, and the surviving class of the second species is identified against a cited theorem
rather than by analogy.

**What is established here, all of it executed below:**

1. On the 12-vertex witness, the entire $2$-dimensional residue $\mathcal H^1_G(\mathcal K_{\rm disj})$
   is read at **full rank by arena triangles** and at **rank 1** by all $1083$ arena 4-cycles.
   Broken out by shape, spectator and rotation squares have *the same image* — one shared line.
2. $\varepsilon_{\rm harm}(\mathcal K_{3,4}) > 0$ **exists**: the cycle graph $C_n$, every $n \ge 5$,
   $K \in \{2,3,4\}$, under the trivial group and under $\mathbb Z_n$. The falsifiable question
   posed against the previous round is answered yes.
3. $L_{\rm closure}(C_n, K) = n$ **exactly** — a clean step function, so the residue is invisible
   to every bounded vocabulary $\mathcal K_{\le L}$ with $L < n$.
4. **The class is identified, not likened.** At trivial $G$, $\varepsilon_{\rm harm}$ *is* simplicial
   $H^1$ over $\mathbb Q$, and $\mathcal K_{\rm disj}$ *is* the 2-cell set of Abrams' discrete
   configuration space $UD^K(X)$. Abrams' hypothesis for $C_n$ reduces to $n \ge K+1$; the measured
   rank is $1$ on every admissible $(n,K)$ and drops to $0$ at $K=n$ where the hypothesis fails.
   The group action on the class is solved exactly: $+1$ for rotations, $-1$ for reflections.
5. **The odd-cycle reading is false.** For even $n$ the arena is *bipartite* — no odd cycle of any
   length — yet the class survives, first detected at even length $n$. Parity was incidental;
   **omission** from the declared family is the operative property.
6. Everything cross-checked by a second engine built on a group-averaging projector, which never
   classifies an orbit and never applies a reversibility rule.

**The dichotomy.** A harmonic residue is either *vocabulary residue* — $L_{\rm closure}$ bounded,
the reader was missing a cycle shape, admit it and the class evaporates — or *traversal residue* —
$L_{\rm closure}$ tracking the substrate, carried by configuration-space topology, unreachable by
any fixed local vocabulary. $L_{\rm closure}$ separates them and is computable. That is the
difference between apparent and genuine flatness, measured.

Two populated regimes, not a classification theorem. Exact rational arithmetic throughout;
no floating point anywhere in this notebook.

## Library

Two independent engines. Nothing imported from outside the standard library.

In [1]:
# =====================================================================
#  SELF-CONTAINED LIBRARY.  Two independent engines, exact rationals.
#  Engine A: orbit basis + explicit reversibility rule (Relation Layer S1.1-1.5)
#  Engine B: group-averaging projector; no orbits, no canonical forms,
#            no reversibility rule -- reversible directions die by
#            representation theory alone.
# =====================================================================
import itertools, sys
from fractions import Fraction
from math import gcd, lcm
from collections import defaultdict, deque

# ---------------------------------------------------------- exact linear algebra
def rank_exact(rows, nc):
    M = [[Fraction(x) for x in r] for r in rows]; rk = 0
    for c in range(nc):
        p = next((i for i in range(rk, len(M)) if M[i][c] != 0), None)
        if p is None: continue
        M[rk], M[p] = M[p], M[rk]; pv = M[rk][c]
        for i in range(len(M)):
            if i != rk and M[i][c] != 0:
                f = M[i][c]/pv; M[i] = [a-f*b for a, b in zip(M[i], M[rk])]
        rk += 1
        if rk == len(M): break
    return rk

def rref(rows, nc):
    M = [[Fraction(x) for x in r] for r in rows]; piv = []; rr = 0
    for c in range(nc):
        p = next((i for i in range(rr, len(M)) if M[i][c] != 0), None)
        if p is None: continue
        M[rr], M[p] = M[p], M[rr]; pv = M[rr][c]; M[rr] = [x/pv for x in M[rr]]
        for i in range(len(M)):
            if i != rr and M[i][c] != 0:
                f = M[i][c]; M[i] = [a-f*b for a, b in zip(M[i], M[rr])]
        piv.append(c); rr += 1
        if rr == len(M): break
    return M[:rr], piv

def nullsp(rows, nc):
    M = [[Fraction(x) for x in r] for r in rows]; piv = {}; rr = 0
    for c in range(nc):
        p = next((i for i in range(rr, len(M)) if M[i][c] != 0), None)
        if p is None: continue
        M[rr], M[p] = M[p], M[rr]; pv = M[rr][c]; M[rr] = [x/pv for x in M[rr]]
        for i in range(len(M)):
            if i != rr and M[i][c] != 0:
                f = M[i][c]; M[i] = [a-f*b for a, b in zip(M[i], M[rr])]
        piv[c] = rr; rr += 1
    return [[(-M[piv[c]][fc] if c in piv else (1 if c == fc else 0)) for c in range(nc)]
            for fc in [c for c in range(nc) if c not in piv]]

def integ(v):
    L = 1
    for x in v: L = lcm(L, Fraction(x).denominator)
    w = [int(Fraction(x)*L) for x in v]; g = 0
    for x in w: g = gcd(g, abs(x))
    return [x//(g or 1) for x in w]

# ---------------------------------------------------------- base graph
def automorphisms(n, E):
    adj = [set() for _ in range(n)]
    for u, v in E: adj[u].add(v); adj[v].add(u)
    deg = [len(a) for a in adj]; out = []
    def bt(p, used):
        i = len(p)
        if i == n: out.append(tuple(p)); return
        for c in range(n):
            if c in used or deg[c] != deg[i]: continue
            if all((c in adj[p[j]]) == (i in adj[j]) for j in range(i)):
                p.append(c); used.add(c); bt(p, used); p.pop(); used.discard(c)
    bt([], set()); return out

def vertex_orbits(n, G):
    seen, orbs = set(), []
    for v in range(n):
        if v in seen: continue
        o = frozenset(g[v] for g in G); orbs.append(o); seen |= o
    return orbs

def triangles(n, E):
    Es = {frozenset(e) for e in E}
    return [t for t in itertools.combinations(range(n), 3)
            if all(frozenset(p) in Es for p in itertools.combinations(t, 2))]

def girth(n, E):
    A = [set() for _ in range(n)]
    for u, v in E: A[u].add(v); A[v].add(u)
    best = 10**9
    for s in range(n):
        d = {s: 0}; par = {s: None}; q = deque([s])
        while q:
            x = q.popleft()
            for y in A[x]:
                if y not in d: d[y] = d[x]+1; par[y] = x; q.append(y)
                elif y != par[x]: best = min(best, d[x]+d[y]+1)
    return best

# ---------------------------------------------------------- arena (S1.1)
def arena(n, E, K):
    Es = {frozenset(e) for e in E}
    verts = [frozenset(c) for c in itertools.combinations(range(n), K)]
    vidx = {S: i for i, S in enumerate(verts)}
    edges = set()
    for S in verts:
        for u in S:
            for w in range(n):
                if w in S or frozenset((u, w)) not in Es: continue
                edges.add(frozenset((vidx[S], vidx[(S-{u}) | {w}])))
    return verts, vidx, sorted(tuple(sorted(e)) for e in edges)

def act_vert(g, S, verts, vidx): return vidx[frozenset(g[x] for x in verts[S])]

def arena_adj(aedges, nv):
    A = [set() for _ in range(nv)]
    for a, b in aedges: A[a].add(b); A[b].add(a)
    return A

def arena_bipartite(aedges, nv):
    A = arena_adj(aedges, nv); col = {}
    for s in range(nv):
        if s in col: continue
        col[s] = 0; q = deque([s])
        while q:
            x = q.popleft()
            for y in A[x]:
                if y not in col: col[y] = 1-col[x]; q.append(y)
                elif col[y] == col[x]: return False
    return True

# ---------------------------------------------------------- ENGINE A
def c1_basis(verts, vidx, aedges, G):
    oriented = [(a, b) for (a, b) in aedges] + [(b, a) for (a, b) in aedges]
    lab = {}; basis = []
    for oe in oriented:
        if oe in lab: continue
        orb = {(act_vert(g, oe[0], verts, vidx), act_vert(g, oe[1], verts, vidx)) for g in G}
        if (oe[1], oe[0]) in orb:                      # reversible orbit -> forced zero
            for x in orb: lab[x] = (None, 0)
        else:
            col = len(basis); basis.append(oe)
            for x in orb: lab[x] = (col, +1)
            for x in orb: lab[(x[1], x[0])] = (col, -1)
    return len(basis), basis, lab

def d0_rows(verts, vidx, G, basis):
    seen, cols = set(), []
    for i in range(len(verts)):
        if i in seen: continue
        orb = frozenset(act_vert(g, i, verts, vidx) for g in G); seen |= orb; cols.append(orb)
    return [[(1 if b in orb else 0)-(1 if a in orb else 0) for (a, b) in basis] for orb in cols]

def cycles_len(aedges, nv, L):
    A = arena_adj(aedges, nv); out = set()
    def ext(p, used):
        if len(p) == L:
            if p[0] in A[p[-1]]:
                c = tuple(p); i = c.index(min(c)); f = c[i:]+c[:i]
                r = tuple(reversed(c)); j = r.index(min(r)); b = r[j:]+r[:j]
                out.add(min(f, b))
            return
        for w in A[p[-1]]:
            if w in used or w < p[0]: continue
            p.append(w); used.add(w); ext(p, used); p.pop(); used.discard(w)
    for s in range(nv): ext([s], {s})
    return sorted(out)

def cells_disj(n, E, K, verts, vidx):
    """S1.3 K_disj: disjoint-support commuting squares."""
    out = set()
    def canon(q):
        rots = [tuple(q[i:]+q[:i]) for i in range(4)]
        r = tuple(reversed(q)); rots += [tuple(r[i:]+r[:i]) for i in range(4)]
        return min(rots)
    for S in verts:
        for e1, e2 in itertools.combinations(E, 2):
            if set(e1) & set(e2): continue
            for u1, w1 in ((e1[0], e1[1]), (e1[1], e1[0])):
                if u1 not in S or w1 in S: continue
                for u2, w2 in ((e2[0], e2[1]), (e2[1], e2[0])):
                    if u2 not in S or w2 in S: continue
                    q = (vidx[S], vidx[(S-{u1}) | {w1}],
                         vidx[(S-{u1, u2}) | {w1, w2}], vidx[(S-{u2}) | {w2}])
                    if len(set(q)) == 4: out.add(canon(q))
    return sorted(out)

def rowof(cyc, lab, dim1):
    row = [0]*dim1
    for i in range(len(cyc)):
        col, sg = lab[(cyc[i], cyc[(i+1) % len(cyc)])]
        if col is not None: row[col] += sg
    return row

def engineA(n, E, K, G=None):
    if G is None: G = automorphisms(n, E)
    verts, vidx, aedges = arena(n, E, K)
    dim1, basis, lab = c1_basis(verts, vidx, aedges, G)
    M0 = d0_rows(verts, vidx, G, basis); r0 = rank_exact(M0, dim1)
    return dict(G=G, verts=verts, vidx=vidx, aedges=aedges, dim1=dim1, basis=basis,
                lab=lab, M0=M0, r0=r0, eps=dim1-r0, nv=len(verts), ne=len(aedges),
                orbits=len(vertex_orbits(n, G)))

def eps_harm(R, rows):
    return R['dim1']-R['r0']-rank_exact(rows, R['dim1'])

# ---------------------------------------------------------- ENGINE B (projector)
def engineB(n, E, K, G, Ls=(4,)):
    Es = {frozenset(e) for e in E}
    AV = sorted(itertools.combinations(range(n), K)); ai = {s: i for i, s in enumerate(AV)}
    AE = set()
    for s in AV:
        for u in s:
            for w in range(n):
                if w in s or frozenset((u, w)) not in Es: continue
                t = tuple(sorted(set(s)-{u} | {w}))
                AE.add((min(ai[s], ai[t]), max(ai[s], ai[t])))
    AE = sorted(AE); ei = {e: i for i, e in enumerate(AE)}
    NV, NE = len(AV), len(AE)
    def gv(g, i): return ai[tuple(sorted(g[x] for x in AV[i]))]
    def gedge(g, k):
        a, b = AE[k]; ga, gb = gv(g, a), gv(g, b)
        return (ei[(ga, gb)], 1) if ga < gb else (ei[(gb, ga)], -1)
    P1 = [[0]*NE for _ in range(NE)]
    for k in range(NE):
        for g in G:
            j, s = gedge(g, k); P1[k][j] += s
    B, _ = rref(P1, NE); dim1 = len(B)
    P0 = [[0]*NV for _ in range(NV)]
    for i in range(NV):
        for g in G: P0[i][gv(g, i)] += 1
    B0, _ = rref(P0, NV)
    D0 = [[f[AE[k][1]]-f[AE[k][0]] for k in range(NE)] for f in B0]
    r0 = rank_exact(D0, NE)
    A2 = {i: set() for i in range(NV)}
    for a, b in AE: A2[a].add(b); A2[b].add(a)
    def cyc(L):
        out = set()
        def ext(p, used):
            if len(p) == L:
                if p[0] in A2[p[-1]]:
                    q = tuple(p); rr = [tuple(q[i:]+q[:i]) for i in range(L)]
                    rq = tuple(reversed(q)); rr += [tuple(rq[i:]+rq[:i]) for i in range(L)]
                    out.add(min(rr))
                return
            for w in A2[p[-1]]:
                if w in used or w < p[0]: continue
                p.append(w); used.add(w); ext(p, used); p.pop(); used.discard(w)
        for s in range(NV): ext([s], {s})
        return sorted(out)
    res, acc = {}, []
    for L in Ls:
        cells = cyc(L)
        for q in cells:
            r = [0]*NE
            for i in range(L):
                p, qq = q[i], q[(i+1) % L]; k = ei[(min(p, qq), max(p, qq))]
                r[k] += 1 if p < qq else -1
            acc.append(r)
        M = [[sum(r[k]*B[j][k] for k in range(NE)) for j in range(dim1)] for r in acc]
        r1 = rank_exact(M, dim1); res[L] = (len(cells), r1, dim1-r0-r1)
    return dict(NV=NV, NE=NE, dim1=dim1, r0=r0, eps=dim1-r0, res=res)

def Cn(k): return k, [(i, (i+1) % k) for i in range(k)]
def rotations(k): return [tuple((i+j) % k for i in range(k)) for j in range(k)]
def trivial(k): return [tuple(range(k))]

WITNESS = (12, [(0,1),(0,2),(0,4),(0,5),(0,6),(0,7),(0,9),(1,2),(1,3),(1,5),(1,7),(1,8),
                (1,10),(2,3),(2,4),(2,8),(2,9),(2,11),(3,4),(3,5),(3,6),(3,9),(3,10),
                (4,5),(4,7),(4,10),(4,11),(5,6),(5,8),(5,11)])
print("library loaded: engineA (orbit/reversibility) + engineB (projector), exact rationals")

library loaded: engineA (orbit/reversibility) + engineB (projector), exact rationals


## 1. What each reader vocabulary can actually read

The 12-vertex witness at $K=2$. $\mathcal H^1_G(\mathcal K_{\rm disj})$ has dimension 2; the question
is the **rank** of each cycle family's circulation map on that 2-dimensional space — not how many
cells it has.

The structural split matters. An arena 4-cycle is one of exactly three things: a *commuting* square
(two independent token moves — this is precisely $\mathcal K_{\rm disj}$), a *spectator* square (one
token parked, the other walking a base 4-cycle), or a *rotation* square (four base edges, no parked
token). Arena 3-cycles split the same way.

In [2]:
# ---- the 12-vertex witness: what each reader vocabulary can actually read ----
n, E = WITNESS; K = 2
R = engineA(n, E, K)
verts, lab, dim1 = R['verts'], R['lab'], R['dim1']
print(f"base graph  {n}v {len(E)}e   girth {girth(n,E)}   {len(triangles(n,E))} triangles"
      f"   |Aut|={len(R['G'])}   {R['orbits']} vertex orbits")
print(f"arena       {R['nv']}v {R['ne']}e   dim C1_G={dim1}  rank d0={R['r0']}  eps={R['eps']}")

fam = {L: cycles_len(R['aedges'], R['nv'], L) for L in (3, 4, 5)}
Cdisj = cells_disj(n, E, K, verts, R['vidx'])
Rdj = [rowof(c, lab, dim1) for c in Cdisj]
Rw  = {L: [rowof(c, lab, dim1) for c in fam[L]] for L in fam}

# H^1_G(K_disj) = ker d1(K_disj) / im d0, as an explicit integral basis
Bc, Pc = rref(R['M0'], dim1); comp = []
for v in nullsp(Rdj, dim1):
    r = [Fraction(x) for x in v]
    for b, c in zip(Bc, Pc):
        if r[c] != 0: f = r[c]; r = [a-f*x for a, x in zip(r, b)]
    nz = next((c for c in range(dim1) if r[c] != 0), None)
    if nz is None: continue
    r = [x/r[nz] for x in r]; comp.append(r); Bc, Pc = rref(Bc+[r], dim1)
H = [integ(v) for v in comp]; m = len(H)
print(f"\nH^1_G(K_disj) = ker d1 ({dim1-rank_exact(Rdj,dim1)}) / im d0 ({R['r0']}) = dim {m}")

def proj(c):
    r = rowof(c, lab, dim1)
    return [sum(r[i]*h[i] for i in range(dim1)) for h in H]

def kind3(c):
    S = [set(verts[x]) for x in c]
    return "spectator  (1 token fixed, other walks a base triangle)" if set.intersection(*S) \
        else "rotation   (3 base edges, no fixed token)"
def kind4(c):
    S = [set(verts[x]) for x in c]; mv = []
    for i in range(4):
        a, b = S[i], S[(i+1) % 4]; mv.append(frozenset(((a-b).pop(), (b-a).pop())))
    if len(set(mv)) == 2: return "commuting  (2 base edges, tokens independent) = K_disj"
    return "spectator  (1 token fixed, other walks a base 4-cycle)" if set.intersection(*S) \
        else "rotation   (4 base edges, no fixed token)"

by3, by4 = defaultdict(list), defaultdict(list)
for c in fam[3]: by3[kind3(c)].append(c)
for c in fam[4]: by4[kind4(c)].append(c)

print(f"\n  READER VOCABULARY  vs  the dim-{m} invariant residue H^1_G(K_disj)")
print("  family                                                      cells  rank d1  rank_H")
def line(tag, cs):
    print(f"  {tag:58s} {len(cs):5d} {rank_exact([rowof(c,lab,dim1) for c in cs],dim1):8d}"
          f" {rank_exact([proj(c) for c in cs],m):7d}")
for k in sorted(by3, key=lambda k: -len(by3[k])): line("3-cycle  "+k, by3[k])
line("3-cycle  ALL arena 3-cycles", fam[3])
print()
for k in sorted(by4, key=lambda k: -len(by4[k])): line("4-cycle  "+k, by4[k])
line("4-cycle  ALL arena 4-cycles", fam[4])
print()
line("5-cycle  ALL arena 5-cycles", fam[5])
print(f"\n  cross-check 1  cells_disj enumeration = {len(Cdisj)} cells, rank d1 = "
      f"{rank_exact(Rdj,dim1)}  -- matches the 'commuting' class above by an independent route")
nt = len(triangles(n, E))
print(f"  cross-check 2  arena 3-cycles predicted {nt}*(n-3) spectator + {nt} rotation "
      f"= {nt*(n-3)+nt}; enumerated {len(fam[3])}")
bad = sum(1 for L in fam for c in fam[L] for col in R['M0']
          if sum(a*b for a, b in zip(rowof(c, lab, dim1), col)) != 0)
print(f"  cross-check 3  im d0 has zero circulation on every arena cycle: "
      f"{'PASS' if bad==0 else 'FAIL'}")

base graph  12v 30e   girth 3   20 triangles   |Aut|=24   2 vertex orbits
arena       66v 300e   dim C1_G=22  rank d0=7  eps=15

H^1_G(K_disj) = ker d1 (9) / im d0 (7) = dim 2

  READER VOCABULARY  vs  the dim-2 invariant residue H^1_G(K_disj)
  family                                                      cells  rank d1  rank_H
  3-cycle  spectator  (1 token fixed, other walks a base triangle)   180       10       2
  3-cycle  rotation   (3 base edges, no fixed token)            20        2       2
  3-cycle  ALL arena 3-cycles                                  200       12       2



  4-cycle  spectator  (1 token fixed, other walks a base 4-cycle)   528        8       1


  4-cycle  commuting  (2 base edges, tokens independent) = K_disj   291       13       0
  4-cycle  rotation   (4 base edges, no fixed token)           264        7       1
  4-cycle  ALL arena 4-cycles                                 1083       14       1



  5-cycle  ALL arena 5-cycles                                 5688       15       2

  cross-check 1  cells_disj enumeration = 291 cells, rank d1 = 13  -- matches the 'commuting' class above by an independent route
  cross-check 2  arena 3-cycles predicted 20*(n-3) spectator + 20 rotation = 200; enumerated 200
  cross-check 3  im d0 has zero circulation on every arena cycle: PASS


Three structurally distinct square shapes; two are nonzero readers; the union still reads **rank 1**.
Meanwhile 20 rotation triangles, one per base triangle, read **rank 2** — the whole residue. The
minimal complete reader is 20 cells of a shape Definition 1.3 excludes, since it demands four
distinct arena vertices.

## 2. Does anything survive both triangles and squares?

Both arena 3-cycle types require a base triangle. So on a triangle-free base $\mathcal K_{3,4}$
collapses to $\mathcal K_4$, and the question becomes whether a triangle-free graph has
$\varepsilon_{\rm harm}(\mathcal K_4) > 0$.

In [3]:
# ---- the falsifiable question: does any class survive BOTH 3- and 4-cells? ----
# Both arena 3-cycle types need a base triangle, so on a triangle-free base
# K_34 collapses to K_4.  Check that first, then hunt there.
print("  triangle-free base  =>  zero arena 3-cycles  (so K_34 = K_4 exactly)")
for n, K in ((5,2),(6,2),(7,2),(9,2),(6,3),(8,3)):
    _, E = Cn(n); verts, vidx, aedges = arena(n, E, K)
    print(f"   C{n} K={K}:  base triangles {len(triangles(n,E))}   "
          f"arena 3-cycles {len(cycles_len(aedges,len(verts),3))}")

print("\n  eps_harm(K_34) = eps_harm(K_4) on C_n, under three groups")
print("  graph  K   G               |G| orb  dim C1_G  r0  eps  #4cyc  rank d1  eps_harm")
rows = []
for n in (5, 6, 7, 8, 9, 10, 11, 12):
    _, E = Cn(n); full = automorphisms(n, E)
    for K in (2, 3):
        if K >= n-1 or len(list(itertools.combinations(range(n), K))) > 130: continue
        for gn, G in (("trivial", trivial(n)), (f"Z_{n} rotations", rotations(n)),
                      ("full dihedral", full)):
            R = engineA(n, E, K, G)
            C4 = cycles_len(R['aedges'], R['nv'], 4)
            r1 = rank_exact([rowof(c, R['lab'], R['dim1']) for c in C4], R['dim1'])
            eh = R['dim1']-R['r0']-r1
            rows.append((n, K, gn, eh))
            print(f"  C{n:<4d} {K}  {gn:14s} {len(G):4d} {R['orbits']:3d} {R['dim1']:8d} "
                  f"{R['r0']:4d} {R['eps']:4d} {len(C4):6d} {r1:8d} {eh:8d}")
print(f"\n  eps_harm(K_34) > 0 in {sum(1 for r in rows if r[3]>0)} of {len(rows)} rows."
      f"  Every trivial-G and every Z_n row gives exactly 1; every full-dihedral row gives 0.")

  triangle-free base  =>  zero arena 3-cycles  (so K_34 = K_4 exactly)
   C5 K=2:  base triangles 0   arena 3-cycles 0
   C6 K=2:  base triangles 0   arena 3-cycles 0
   C7 K=2:  base triangles 0   arena 3-cycles 0
   C9 K=2:  base triangles 0   arena 3-cycles 0
   C6 K=3:  base triangles 0   arena 3-cycles 0
   C8 K=3:  base triangles 0   arena 3-cycles 0

  eps_harm(K_34) = eps_harm(K_4) on C_n, under three groups
  graph  K   G               |G| orb  dim C1_G  r0  eps  #4cyc  rank d1  eps_harm
  C5    2  trivial           1   5       15    9    6      5        5        1
  C5    2  Z_5 rotations     5   1        3    1    2      5        1        1
  C5    2  full dihedral    10   1        1    1    0      5        0        0
  C5    3  trivial           1   5       15    9    6      5        5        1
  C5    3  Z_5 rotations     5   1        3    1    2      5        1        1
  C5    3  full dihedral    10   1        1    1    0      5        0        0
  C6    2  trivial      

  C7    3  trivial           1   7       70   34   36     42       35        1
  C7    3  Z_7 rotations     7   1       10    4    6     42        5        1
  C7    3  full dihedral    14   1        4    3    1     42        1        0
  C8    2  trivial           1   8       48   27   21     20       20        1
  C8    2  Z_8 rotations     8   1        6    3    3     20        2        1
  C8    2  full dihedral    16   1        3    3    0     20        0        0
  C8    3  trivial           1   8      120   55   65     80       64        1
  C8    3  Z_8 rotations     8   1       15    6    9     80        8        1
  C8    3  full dihedral    16   1        6    4    2     80        2        0


  C9    2  trivial           1   9       63   35   28     27       27        1
  C9    2  Z_9 rotations     9   1        7    3    4     27        3        1
  C9    2  full dihedral    18   1        3    3    0     27        0        0


  C9    3  trivial           1   9      189   83  106    135      105        1
  C9    3  Z_9 rotations     9   1       21    9   12    135       11        1
  C9    3  full dihedral    18   1        9    6    3    135        3        0
  C10   2  trivial           1  10       80   44   36     35       35        1
  C10   2  Z_10 rotations   10   1        8    4    4     35        3        1
  C10   2  full dihedral    20   1        4    4    0     35        0        0


  C10   3  trivial           1  10      280  119  161    210      160        1
  C10   3  Z_10 rotations   10   1       28   11   17    210       16        1
  C10   3  full dihedral    20   1       12    7    5    210        5        0
  C11   2  trivial           1  11       99   54   45     44       44        1
  C11   2  Z_11 rotations   11   1        9    4    5     44        4        1
  C11   2  full dihedral    22   1        4    4    0     44        0        0


  C12   2  trivial           1  12      120   65   55     54       54        1
  C12   2  Z_12 rotations   12   1       10    5    5     54        4        1
  C12   2  full dihedral    24   1        5    5    0     54        0        0

  eps_harm(K_34) > 0 in 28 of 42 rows.  Every trivial-G and every Z_n row gives exactly 1; every full-dihedral row gives 0.


$\varepsilon_{\rm harm}(\mathcal K_{3,4}) = 1$ on the simplest nontrivial graph there is.

Note which group kills it. Trivial $G$ and the rotation subgroup $\mathbb Z_n$ both carry the class;
the **full dihedral group destroys it**. Section 4 computes why.

## 3. What the surviving class does, and how deep you must go to see it

In [4]:
# ---- what the surviving class IS: track the tokens around its generator ----
def shortest_detector(R, rows_killed, Lmax=14):
    """shortest arena cycle with nonzero circulation on a surviving class"""
    H = nullsp(rows_killed + R['M0'], R['dim1'])
    if not H: return None, None, None
    h = H[0]; lab = R['lab']; A = arena_adj(R['aedges'], R['nv'])
    def circ(c):
        t = Fraction(0)
        for i in range(len(c)):
            col, sg = lab[(c[i], c[(i+1) % len(c)])]
            if col is not None: t += sg*h[col]
        return t
    for L in range(3, Lmax+1):
        hit = [None]
        def ext(p, used):
            if hit[0]: return
            if len(p) == L:
                if p[0] in A[p[-1]]:
                    v = circ(tuple(p))
                    if v != 0: hit[0] = (tuple(p), v)
                return
            for w in A[p[-1]]:
                if w in used or w < p[0]: continue
                p.append(w); used.add(w); ext(p, used); p.pop(); used.discard(w)
        for s in range(R['nv']):
            ext([s], {s})
            if hit[0]: break
        if hit[0]: return L, hit[0][0], hit[0][1]
    return None, None, None

print("  GATE  the generator of the surviving C_n class exchanges the two tokens")
print("  graph  loop len   loop                                      token map")
for n in (5, 6, 7, 8):
    _, E = Cn(n); R = engineA(n, E, 2, rotations(n))
    C4 = cycles_len(R['aedges'], R['nv'], 4)
    L, c, v = shortest_detector(R, [rowof(x, R['lab'], R['dim1']) for x in C4])
    seq = [sorted(R['verts'][x]) for x in c]
    cur = {seq[0][0]: 'A', seq[0][1]: 'B'}
    for i in range(len(seq)):
        a, b = set(seq[i]), set(seq[(i+1) % len(seq)])
        u = (a-b).pop(); w = (b-a).pop(); cur[w] = cur.pop(u)
    sw = cur != {seq[0][0]: 'A', seq[0][1]: 'B'}
    print(f"  C{n:<4d} {L:7d}   {str(seq)[:42]:42s}  {cur}  "
          f"{'EXCHANGED' if sw else 'identity'}")

print("\n  GATE  L_closure: eps_harm(K_<=L) as the declared cycle length grows")
print("  graph  K        " + "".join(f"L<={L:<4d}" for L in range(3, 13)) + "  L_closure")
for n, K in ((5,2),(6,2),(7,2),(8,2),(9,2),(10,2),(6,3),(7,3)):
    _, E = Cn(n); R = engineA(n, E, K, rotations(n))
    acc, out, Lc = [], [], None
    for L in range(3, 13):
        acc += [rowof(c, R['lab'], R['dim1']) for c in cycles_len(R['aedges'], R['nv'], L)]
        e = R['dim1']-R['r0']-rank_exact(acc, R['dim1']); out.append(e)
        if e == 0 and Lc is None: Lc = L
    print(f"  C{n:<4d} {K}        " + "".join(f"{x:<8d}" for x in out) + f"  {Lc}  (n={n})")
print("\n  L_closure = n exactly, for every row.  Unbounded in the substrate size.")

  GATE  the generator of the surviving C_n class exchanges the two tokens
  graph  loop len   loop                                      token map
  C5          5   [[0, 1], [0, 2], [0, 3], [0, 4], [1, 4]]    {1: 'A', 0: 'B'}  EXCHANGED
  C6          6   [[0, 1], [1, 5], [0, 5], [0, 4], [0, 3], [  {0: 'B', 1: 'A'}  EXCHANGED
  C7          7   [[0, 1], [0, 2], [0, 3], [0, 4], [1, 4], [  {1: 'A', 0: 'B'}  EXCHANGED
  C8          8   [[0, 1], [0, 2], [0, 3], [1, 3], [1, 4], [  {1: 'A', 0: 'B'}  EXCHANGED

  GATE  L_closure: eps_harm(K_<=L) as the declared cycle length grows
  graph  K        L<=3   L<=4   L<=5   L<=6   L<=7   L<=8   L<=9   L<=10  L<=11  L<=12    L_closure
  C5    2        2       1       0       0       0       0       0       0       0       0         5  (n=5)
  C6    2        2       1       1       0       0       0       0       0       0       0         6  (n=6)
  C7    2        3       1       1       1       0       0       0       0       0       0         7  (n=7)

  C8    2        3       1       1       1       1       0       0       0       0       0         8  (n=8)


  C9    2        4       1       1       1       1       1       0       0       0       0         9  (n=9)


  C10   2        4       1       1       1       1       1       1       0       0       0         10  (n=10)


  C6    3        3       1       1       0       0       0       0       0       0       0         6  (n=6)


  C7    3        6       1       1       1       0       0       0       0       0       0         7  (n=7)

  L_closure = n exactly, for every row.  Unbounded in the substrate size.


The generator exchanges the two tokens, and $L_{\rm closure} = n$ on the nose — flat at 1 from
$L=4$ through $L=n-1$, then 0.

## 4. Identifying the class against a cited theorem

Token exchange *suggests* the Möbius half-twist of $\mathrm{UConf}_2(S^1)$. Suggestion is not
identification, so here is the chain that makes it checkable.

At trivial $G$ the framework's own quantity is already the topological one:
$\varepsilon_{\rm harm} = \dim C^1 - \operatorname{rank} d_0 - \operatorname{rank} d_1
= \dim \ker d_1 - \dim \operatorname{im} d_0$, which is simplicial $H^1$ of the 2-complex over
$\mathbb Q$. And $\mathcal K_{\rm disj}$ — vertices $=$ $K$-subsets, edges $=$ single token moves,
2-cells $=$ pairs of base edges with disjoint closed supports — is exactly the cell structure of
Abrams' **discrete configuration space** $UD^K(X)$. So no analogy is involved in the first step:
$\varepsilon_{\rm harm}(\mathcal K_{\rm disj} \mid G \text{ trivial}) = \dim H^1(UD^K(X); \mathbb Q)$
by definition.

**Abrams' Theorem A**, as stated by Prue–Scrimshaw: for a graph $G$ with at least $n$ vertices,
$\mathcal C^n(G)$ deformation retracts onto $\mathcal D^n(G)$ if **(A′)** each path connecting
distinct essential vertices has length at least $n+1$, and **(B′)** each homotopically essential
path connecting a vertex to itself has length at least $n+1$. (Prue–Scrimshaw improve (A′) to
$n-1$ via discrete Morse theory; (B′) is unchanged.)

For $C_n$ every vertex has degree 2, so there are **no essential vertices** and (A′) is vacuous —
the improvement is irrelevant here. The only homotopically essential loop is the whole cycle, of
length $n$, so **(B′) reduces to $n \ge K+1$**. That is a hypothesis with a boundary, which makes it
a test rather than a decoration: the identification must hold across the admissible range and must
**fail** at $K=n$.

In [5]:
# ---- the topological identification, run rather than asserted ----
# At trivial G, eps_harm = dim C1 - rank d0 - rank d1 = dim ker d1 - dim im d0,
# which IS simplicial H^1 of the 2-complex over Q.  And the K_disj 2-cells are
# exactly Abrams' discrete configuration space UD^K(X).  So this is not an
# analogy: eps_harm(K_disj | trivial G) = dim H^1(UD^K(X); Q) by definition.
# Abrams' (B') for C_n reduces to n >= K+1.  Test the whole range AND the boundary.
print("  eps_harm(K_disj | G trivial) = dim H^1(UD^K(C_n); Q).   Abrams (B'): n >= K+1")
print("  C_n   K   n>=K+1   arena_v  arena_e   measured   H^1(UConf_K(S^1))   agree")
bad = 0
for n in (3, 4, 5, 6, 7, 8, 9, 10):
    _, E = Cn(n)
    for K in range(2, n+1):
        if len(list(itertools.combinations(range(n), K))) > 140: continue
        R = engineA(n, E, K, trivial(n))
        Cd = cells_disj(n, E, K, R['verts'], R['vidx'])
        r1 = rank_exact([rowof(c, R['lab'], R['dim1']) for c in Cd], R['dim1'])
        eh = R['dim1']-R['r0']-r1
        hyp = n >= K+1
        ok = (eh == 1) == hyp
        bad += not ok
        print(f"  C{n:<4d}{K:3d}   {'yes' if hyp else 'NO ':6s} {R['nv']:8d} {R['ne']:8d}"
              f" {eh:10d} {1:19d}   {'OK' if ok else 'MISMATCH'}")
print(f"\n  {bad} disagreements.  Rank 1 exactly when Abrams' hypothesis holds; 0 at K=n,")
print("  where every vertex is occupied, no token can move, and the discrete model")
print("  collapses to a point while UConf_n(S^1) is still ~ S^1.  The bound is sharp here.")

# ---- the equivariant half: what does the group DO to the class? ----
def solve_exact(cols, rhs, nc):
    m = len(cols)
    M = [[Fraction(cols[j][i]) for j in range(m)]+[Fraction(rhs[i])] for i in range(nc)]
    piv = {}; rr = 0
    for c in range(m):
        p = next((i for i in range(rr, nc) if M[i][c] != 0), None)
        if p is None: continue
        M[rr], M[p] = M[p], M[rr]; pv = M[rr][c]; M[rr] = [x/pv for x in M[rr]]
        for i in range(nc):
            if i != rr and M[i][c] != 0:
                f = M[i][c]; M[i] = [a-f*b for a, b in zip(M[i], M[rr])]
        piv[c] = rr; rr += 1
    for i in range(rr, nc):
        if M[i][m] != 0: return None
    x = [Fraction(0)]*m
    for c, r in piv.items(): x[c] = M[r][m]
    return x

print("\n  solve  g*h = lambda*h + d0 f  exactly, for the spanning class h")
print("  C_n  K   element                     lambda  solvable   H^1 invariants")
for n in (5, 6, 7, 8, 9):
    _, E = Cn(n)
    for K in (2, 3):
        if K >= n: continue
        R = engineA(n, E, K, trivial(n))
        Cd = cells_disj(n, E, K, R['verts'], R['vidx'])
        M1 = [rowof(c, R['lab'], R['dim1']) for c in Cd]
        Bc, Pc = rref(R['M0'], R['dim1']); comp = []
        for v in nullsp(M1, R['dim1']):
            r = [Fraction(x) for x in v]
            for b, c in zip(Bc, Pc):
                if r[c] != 0: f = r[c]; r = [a-f*x for a, x in zip(r, b)]
            nz = next((c for c in range(R['dim1']) if r[c] != 0), None)
            if nz is None: continue
            r = [x/r[nz] for x in r]; comp.append(r); Bc, Pc = rref(Bc+[r], R['dim1'])
        h = [Fraction(x) for x in integ(comp[0])]
        cols = [h]+[[Fraction(x) for x in row] for row in R['M0']]
        for nm, g in (("rotation   i -> i+1", tuple((i+1) % n for i in range(n))),
                      ("reflection i -> -i",  tuple((-i) % n for i in range(n)))):
            gh = [Fraction(0)]*R['dim1']
            for col, (a, b) in enumerate(R['basis']):
                ga = act_vert(g, a, R['verts'], R['vidx'])
                gb = act_vert(g, b, R['verts'], R['vidx'])
                c2, s2 = R['lab'][(ga, gb)]
                if c2 is not None: gh[col] = s2*h[c2]
            x = solve_exact(cols, gh, R['dim1'])
            inv = "rank 1 (acts trivially)" if x and x[0] == 1 else \
                  "0 (class = -class)" if x and x[0] == -1 else "?"
            print(f"  C{n:<3d} {K}   {nm:26s} {str(x[0]) if x else '-':>6s}  "
                  f"{'yes' if x else 'NO':8s}   {inv}")
print("\n  Rotations fix the class, reflections negate it.  So H^1 invariants are")
print("  rank 1 under Z_n and 0 under the full dihedral group -- which is exactly")
print("  the dihedral column measured in section 2, now explained rather than noted.")

  eps_harm(K_disj | G trivial) = dim H^1(UD^K(C_n); Q).   Abrams (B'): n >= K+1
  C_n   K   n>=K+1   arena_v  arena_e   measured   H^1(UConf_K(S^1))   agree
  C3     2   yes           3        3          1                   1   OK
  C3     3   NO            1        0          0                   1   OK
  C4     2   yes           6        8          1                   1   OK
  C4     3   yes           4        4          1                   1   OK
  C4     4   NO            1        0          0                   1   OK
  C5     2   yes          10       15          1                   1   OK
  C5     3   yes          10       15          1                   1   OK
  C5     4   yes           5        5          1                   1   OK
  C5     5   NO            1        0          0                   1   OK
  C6     2   yes          15       24          1                   1   OK
  C6     3   yes          20       36          1                   1   OK
  C6     4   yes          15 

  C8     3   yes          56      120          1                   1   OK


  C8     4   yes          70      160          1                   1   OK
  C8     5   yes          56      120          1                   1   OK
  C8     6   yes          28       48          1                   1   OK
  C8     7   yes           8        8          1                   1   OK
  C8     8   NO            1        0          0                   1   OK
  C9     2   yes          36       63          1                   1   OK


  C9     3   yes          84      189          1                   1   OK


  C9     4   yes         126      315          1                   1   OK


  C9     5   yes         126      315          1                   1   OK


... (2 outputs left out)

  C10    7   yes         120      280          1                   1   OK
  C10    8   yes          45       80          1                   1   OK
  C10    9   yes          10       10          1                   1   OK
  C10   10   NO            1        0          0                   1   OK

  0 disagreements.  Rank 1 exactly when Abrams' hypothesis holds; 0 at K=n,
  where every vertex is occupied, no token can move, and the discrete model
  collapses to a point while UConf_n(S^1) is still ~ S^1.  The bound is sharp here.

  solve  g*h = lambda*h + d0 f  exactly, for the spanning class h
  C_n  K   element                     lambda  solvable   H^1 invariants
  C5   2   rotation   i -> i+1             1  yes        rank 1 (acts trivially)
  C5   2   reflection i -> -i             -1  yes        0 (class = -class)
  C5   3   rotation   i -> i+1             1  yes        rank 1 (acts trivially)
  C5   3   reflection i -> -i             -1  yes        0 (class = -class)
  C6   2   ro

  C7   3   rotation   i -> i+1             1  yes        rank 1 (acts trivially)
  C7   3   reflection i -> -i             -1  yes        0 (class = -class)
  C8   2   rotation   i -> i+1             1  yes        rank 1 (acts trivially)
  C8   2   reflection i -> -i             -1  yes        0 (class = -class)


  C8   3   rotation   i -> i+1             1  yes        rank 1 (acts trivially)
  C8   3   reflection i -> -i             -1  yes        0 (class = -class)
  C9   2   rotation   i -> i+1             1  yes        rank 1 (acts trivially)
  C9   2   reflection i -> -i             -1  yes        0 (class = -class)


  C9   3   rotation   i -> i+1             1  yes        rank 1 (acts trivially)
  C9   3   reflection i -> -i             -1  yes        0 (class = -class)

  Rotations fix the class, reflections negate it.  So H^1 invariants are
  rank 1 under Z_n and 0 under the full dihedral group -- which is exactly
  the dihedral column measured in section 2, now explained rather than noted.


Rank 1 exactly on the admissible range, 0 at $K=n$ where (B′) fails, and the group action solved
exactly: rotations fix the class, reflections negate it. The dihedral column of section 2 is
therefore not an observation, it is $H^1$ invariants of a rank-1 space carrying a $-1$ action.

What is cited and what is run, kept apart: the homotopy equivalence $UD^K \simeq \mathrm{UConf}_K$
is Abrams' theorem, used here with its hypothesis checked, not reproved; the descent of the
retraction from ordered to unordered configurations is the standard free-quotient argument;
$\mathrm{UConf}_K(S^1) \simeq S^1$ is classical. What this notebook contributes is the measured
rank across the whole admissible range, the sharpness at the boundary, and the exact equivariant
degree. An explicit chain-level isomorphism is still not written down.

## 5. The odd-cycle reading, refuted

If every surviving class were detected by an arena *odd* cycle, an arena with no odd cycle could not
carry one.

In [6]:
# ---- kill the "odd cycle blind spot" reading outright ----
# If the surviving class were always detected by an arena ODD cycle, then an
# arena with NO odd cycle at all could not carry one.  Test that.
print("  claim under test:  every K_4-harmonic class is detected by an arena odd cycle")
print("  graph  K  arena bipartite?  odd arena cycles  eps_harm(K_4)  L_closure  parity")
for n, K in ((6,2),(8,2),(10,2),(12,2),(6,3),(8,3)):
    _, E = Cn(n); R = engineA(n, E, K, rotations(n))
    if R['nv'] > 130: continue
    bip = arena_bipartite(R['aedges'], R['nv'])
    C4 = cycles_len(R['aedges'], R['nv'], 4)
    r1 = rank_exact([rowof(c, R['lab'], R['dim1']) for c in C4], R['dim1'])
    eh = R['dim1']-R['r0']-r1
    acc, Lc = [], None
    for L in range(3, n+2):
        acc += [rowof(c, R['lab'], R['dim1']) for c in cycles_len(R['aedges'], R['nv'], L)]
        if R['dim1']-R['r0']-rank_exact(acc, R['dim1']) == 0: Lc = L; break
    print(f"  C{n:<4d} {K}  {str(bip):16s} {'NONE' if bip else 'some':17s} {eh:13d}"
          f"  {str(Lc):9s}  {'even' if Lc and Lc%2==0 else 'odd'}")
print("\n  For even n the arena is bipartite: it contains no odd cycle of ANY length,")
print("  yet eps_harm(K_4) = 1.  The class is first detected at L = n, which is even.")
print("  => 'odd-cycle blind spot' is FALSE.  The operative property is OMISSION from")
print("     the declared cell family, not parity.")

  claim under test:  every K_4-harmonic class is detected by an arena odd cycle
  graph  K  arena bipartite?  odd arena cycles  eps_harm(K_4)  L_closure  parity
  C6    2  True             NONE                          1  6          even
  C8    2  True             NONE                          1  8          even


  C10   2  True             NONE                          1  10         even


  C12   2  True             NONE                          1  12         even
  C6    3  True             NONE                          1  6          even


  C8    3  True             NONE                          1  8          even

  For even n the arena is bipartite: it contains no odd cycle of ANY length,
  yet eps_harm(K_4) = 1.  The class is first detected at L = n, which is even.
  => 'odd-cycle blind spot' is FALSE.  The operative property is OMISSION from
     the declared cell family, not parity.


Bipartite arena, zero odd cycles of any length, class still there. The parity reading is dead —
not renamed, refuted.

## 6. Companion engine

Engine A builds the invariant cochain space from edge orbits and an explicit reversibility rule.
Engine B takes the image of $P = \sum_g T_g$ on full oriented-edge coordinates; reversible
directions are annihilated as a representation fact, never classified. If Engine A's bookkeeping
were wrong these numbers would part company.

In [7]:
# ---- companion engine: projector construction, shares no logic with engine A ----
print("  ENGINE B (group-averaging projector)  vs  ENGINE A (orbit basis + reversibility rule)")
print("  graph  K   G        NV   NE   dim C1_G      r0        eps      #4cyc    rank d1   eps_harm")
agree = disagree = 0
for n in (5, 6, 7, 8, 9, 10):
    _, E = Cn(n)
    for K in (2, 3):
        if K >= n-1 or len(list(itertools.combinations(range(n), K))) > 130: continue
        for gn, G in (("triv", trivial(n)), (f"Z{n}", rotations(n))):
            A = engineA(n, E, K, G)
            C4 = cycles_len(A['aedges'], A['nv'], 4)
            a_r1 = rank_exact([rowof(c, A['lab'], A['dim1']) for c in C4], A['dim1'])
            a = (A['nv'], A['ne'], A['dim1'], A['r0'], A['eps'], len(C4), a_r1,
                 A['dim1']-A['r0']-a_r1)
            B = engineB(n, E, K, G, Ls=(4,)); nc, b_r1, b_eh = B['res'][4]
            b = (B['NV'], B['NE'], B['dim1'], B['r0'], B['eps'], nc, b_r1, b_eh)
            ok = a == b
            agree += ok; disagree += (not ok)
            cells = "  ".join(f"{x:>3d}={y:<3d}" if x == y else f"{x}!={y}" for x, y in zip(a, b))
            print(f"  C{n:<4d} {K}  {gn:5s} {cells}   {'OK' if ok else 'MISMATCH'}")
print(f"\n  {agree} rows agree on all eight quantities, {disagree} disagree.")
print("  Engine B never classifies an orbit and never applies a reversibility test:")
print("  reversible directions are annihilated by P = sum_g T_g as a representation fact.")

  ENGINE B (group-averaging projector)  vs  ENGINE A (orbit basis + reversibility rule)
  graph  K   G        NV   NE   dim C1_G      r0        eps      #4cyc    rank d1   eps_harm
  C5    2  triv   10=10    15=15    15=15     9=9      6=6      5=5      5=5      1=1     OK
  C5    2  Z5     10=10    15=15     3=3      1=1      2=2      5=5      1=1      1=1     OK
  C5    3  triv   10=10    15=15    15=15     9=9      6=6      5=5      5=5      1=1     OK
  C5    3  Z5     10=10    15=15     3=3      1=1      2=2      5=5      1=1      1=1     OK
  C6    2  triv   15=15    24=24    24=24    14=14    10=10     9=9      9=9      1=1     OK
  C6    2  Z6     15=15    24=24     4=4      2=2      2=2      9=9      1=1      1=1     OK
  C6    3  triv   20=20    36=36    36=36    19=19    17=17    18=18    16=16     1=1     OK
  C6    3  Z6     20=20    36=36     6=6      3=3      3=3     18=18     2=2      1=1     OK
  C7    2  triv   21=21    35=35    35=35    20=20    15=15    14=14    14=

  C7    3  triv   35=35    70=70    70=70    34=34    36=36    42=42    35=35     1=1     OK
  C7    3  Z7     35=35    70=70    10=10     4=4      6=6     42=42     5=5      1=1     OK
  C8    2  triv   28=28    48=48    48=48    27=27    21=21    20=20    20=20     1=1     OK
  C8    2  Z8     28=28    48=48     6=6      3=3      3=3     20=20     2=2      1=1     OK


  C8    3  triv   56=56   120=120  120=120   55=55    65=65    80=80    64=64     1=1     OK


  C8    3  Z8     56=56   120=120   15=15     6=6      9=9     80=80     8=8      1=1     OK


  C9    2  triv   36=36    63=63    63=63    35=35    28=28    27=27    27=27     1=1     OK
  C9    2  Z9     36=36    63=63     7=7      3=3      4=4     27=27     3=3      1=1     OK


  C9    3  triv   84=84   189=189  189=189   83=83   106=106  135=135  105=105    1=1     OK


  C9    3  Z9     84=84   189=189   21=21     9=9     12=12   135=135   11=11     1=1     OK


  C10   2  triv   45=45    80=80    80=80    44=44    36=36    35=35    35=35     1=1     OK
  C10   2  Z10    45=45    80=80     8=8      4=4      4=4     35=35     3=3      1=1     OK


  C10   3  triv  120=120  280=280  280=280  119=119  161=161  210=210  160=160    1=1     OK


  C10   3  Z10   120=120  280=280   28=28    11=11    17=17   210=210   16=16     1=1     OK

  24 rows agree on all eight quantities, 0 disagree.
  Engine B never classifies an orbit and never applies a reversibility test:
  reversible directions are annihilated by P = sum_g T_g as a representation fact.


## 7. The result

In [8]:
# ---- the two species, side by side ----
def ladder(R, upto):  # stops at the first length that closes everything
    acc, out, Lc = [], {}, None
    for L in range(3, upto+1):
        acc += [rowof(c, R['lab'], R['dim1']) for c in cycles_len(R['aedges'], R['nv'], L)]
        e = R['dim1']-R['r0']-rank_exact(acc, R['dim1']); out[L] = e
        if e == 0: Lc = L; break
    return out, Lc

def prof(nm, n, E, K, G):
    R = engineA(n, E, K, G)
    Cd = cells_disj(n, E, K, R['verts'], R['vidx'])
    f = lambda cs: R['dim1']-R['r0']-rank_exact([rowof(c, R['lab'], R['dim1']) for c in cs], R['dim1'])
    C3 = cycles_len(R['aedges'], R['nv'], 3)
    C4 = cycles_len(R['aedges'], R['nv'], 4)
    lad, Lc = ladder(R, n+1)
    return dict(nm=nm, tri=len(triangles(n, E)), eps=R['eps'], disj=f(Cd), k3=f(C3),
                k4=f(C4), k34=f(C3+C4), Lc=Lc, n=n)

cols = [prof("12v witness", *WITNESS, 2, None),
        prof("C7  (Z_7)", 7, Cn(7)[1], 2, rotations(7)),
        prof("C9  (Z_9)", 9, Cn(9)[1], 2, rotations(9)),
        prof("C10 (Z_10)", 10, Cn(10)[1], 2, rotations(10))]

print("  TWO SPECIES OF INVARIANT HARMONIC RESIDUE          K = 2 throughout")
print("  " + "-"*74)
print("                                " + "".join(f"{c['nm']:>14s}" for c in cols))
for key, label in (("tri", "base triangles"), ("eps", "eps = dim C1_G - rank d0"),
                   ("disj", "eps_harm(K_disj)"), ("k3", "eps_harm(K_3)   triangles only"),
                   ("k4", "eps_harm(K_4)   squares only"), ("k34", "eps_harm(K_34)  both"),
                   ("Lc", "L_closure")):
    print(f"  {label:30s}" + "".join(f"{c[key]:>14d}" for c in cols))
print("  " + "-"*74)
print("""
  Read the K_3 / K_4 / K_34 rows carefully -- the families are not nested.
  On the witness, triangles ALONE leave 3 of the 15 open and squares ALONE leave
  1; only the union closes it, so L_closure = 4.  The triangles are nonetheless
  the complete reader of the 2-dimensional K_disj residue (rank 2 of 2), while
  all 1083 squares read only rank 1 of it.  That is the rank deficiency.

  On C_n there are no triangles at all, so K_34 = K_4 and nothing the 3- or
  4-cycle vocabulary can say closes the class.  L_closure = n.

  VOCABULARY RESIDUE   L_closure bounded (4 here, whatever the substrate size).
      The declared cell family was missing a cycle shape the substrate has.
      Admit the shape and the class is gone; the substrate never moved.

  TRAVERSAL RESIDUE    L_closure = n, unbounded.  No fixed vocabulary K_<=L
      reads it once n > L.  It is H^1 of the configuration space itself
      (UConf_K(S^1) ~ S^1) and its generator is the exchange of two
      indistinguishable tokens -- the Mobius half-twist.  Only the full
      dihedral group kills it, by reversing its orientation.

  L_closure separates them and is computable.  Bounded L_closure says the
  flatness was apparent.  L_closure tracking the substrate says it was genuine:
  there is no shorter address to reach it by.""")

  TWO SPECIES OF INVARIANT HARMONIC RESIDUE          K = 2 throughout
  --------------------------------------------------------------------------
                                   12v witness     C7  (Z_7)     C9  (Z_9)    C10 (Z_10)
  base triangles                            20             0             0             0
  eps = dim C1_G - rank d0                  15             3             4             4
  eps_harm(K_disj)                           2             1             1             1
  eps_harm(K_3)   triangles only             3             3             4             4
  eps_harm(K_4)   squares only               1             1             1             1
  eps_harm(K_34)  both                       0             1             1             1
  L_closure                                  4             7             9            10
  --------------------------------------------------------------------------

  Read the K_3 / K_4 / K_34 rows carefully -- the families are 

## What this changes

**The $\S19$ refutation stands and its mechanism is now typed.** The surviving class there is
vocabulary residue: $L_{\rm closure} = 4$, closed by admitting one excluded cell shape. The graph
never had a defect. The claim to make about it is about the *reader*, and it is narrow — the
4-cycle vocabulary is rank-deficient against this substrate's residue, reading one line of two
with all three of its shapes — not a claim about odd cycles.

**The operative variable is rank access, not cell count, length, or parity.** 1083 squares supply
one independent constraint on the residue; 20 triangles supply two. A relation family's useful
content is the rank of its evaluation map on the surviving sector.

**Vocabulary residue is not the only kind, and the other kind cannot be legislated away.** $C_n$
carries a class no $\mathcal K_{\le L}$ reads for $n > L$. Enriching the cell family is a repair
exactly when $L_{\rm closure}$ is bounded.

**The traversal class is pinned down.** Not "behaves like" the configuration-space generator: it is
$H^1$ of Abrams' discrete configuration space by definition, matched to $\mathrm{UConf}_K(S^1)$
through a theorem whose hypothesis is verified, confirmed across the admissible range, sharp at the
boundary, with its equivariant degree computed. The group matters as much as the vocabulary — the
same substrate and the same cells give rank 1 under $\mathbb Z_n$ and 0 under $D_{2n}$.

**Open, and now sharply posed — the bounded-closure problem.** Give structural conditions on a
family $(X,K,G)$ under which every invariant harmonic residue is annihilated by a vocabulary of
uniformly bounded cycle length. $C_n$ shows the obstruction is real, with
$\sup_n L_{\rm closure}(C_n,K) = \infty$, so any bounded-closure theorem must exclude or quotient
out the configuration-space traversal component. Whether what remains has a uniform bound is
unsettled, and it is a question about token arenas, not about this one witness.

**Sources.** Abrams' theorem and the improved subdivision bound:
[Prue & Scrimshaw, *Abrams's stable equivalence for graph braid groups*](https://arxiv.org/abs/0909.5511).
Background on discretized graph configuration spaces:
[Abrams & Ghrist, *Finding topology in a factory: configuration spaces*](https://arxiv.org/html/math/0009118v1).
Later work on the stabilization bound:
[Abrams' stabilization theorem for no-$k$-equal configuration spaces on graphs](https://arxiv.org/abs/2407.07854).